# Multivariate EDA

## Imports

In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from sklearn.preprocessing import LabelEncoder

## Data Preparation

In [ ]:
data_df = pd.read_csv("../data/sample.csv")
column_nature_df = pd.read_csv("../data/column_nature.sample.csv")
categorical_features_df = column_nature_df[column_nature_df["columnNature"] == "categorical"]
categorical_features = categorical_features_df["columnName"].to_list()

label_encoder = LabelEncoder()
encoded_data_df = data_df.copy()

non_singular_columns = []
for column in encoded_data_df.columns:
    if encoded_data_df[column].value_counts().size > 1:
        non_singular_columns.append(column)

encoded_data_df = encoded_data_df.filter(non_singular_columns)
encoded_alive_data_df = data_df.copy().filter(non_singular_columns)
encoded_alive_data_df = encoded_alive_data_df[encoded_alive_data_df["Outcome"] == "Alive"]

ordinal_columns = ["BirthMultiplicity_term","BirthFinalModeDeliver_term"]

# encode categorical data
for feature in categorical_features:
    if feature in non_singular_columns:
        encoded_data_df[feature] = label_encoder.fit_transform(encoded_data_df[feature])
        encoded_alive_data_df[feature] = label_encoder.fit_transform(encoded_alive_data_df[feature])

In [ ]:
encoded_data_df.shape

In [ ]:
def pvalue_continuous_vs_binary(continuous, binary, equal_var=False):
    """
    Compute p-value comparing a continuous variable across a binary group.

    Parameters
    ----------
    continuous : pandas Series
        Continuous variable (e.g. maternal age)

    binary : pandas Series
        Binary variable (0/1)

    equal_var : bool
        Whether to assume equal variance for t-test

    Returns
    -------
    t-statistic : float
        The calculated t-statistic for the test.
    p-value : float
        The p-value for the test.
    """

    group0 = continuous[binary == 0]
    group1 = continuous[binary == 1]
    
    t_stat, p_value = stats.ttest_ind(group0, group1, equal_var=equal_var)

    return t_stat, p_value

In [ ]:
def numeric_correlation(series1, series2, method="pearson"):
    """
    Compute correlation and p-value between two numeric variables.

    Parameters
    ----------
    series1 : pandas Series
    series2 : pandas Series
    method : str
        "pearson" or "spearman"

    Returns
    -------
    dict with correlation coefficient and p-value
    """

    df = pd.DataFrame({
        "x": series1,
        "y": series2
    }).dropna()

    x = df["x"]
    y = df["y"]

    if method == "pearson":
        r, p = stats.pearsonr(x, y)
        method_name = "Pearson correlation"

    elif method == "spearman":
        r, p = stats.spearmanr(x, y)
        method_name = "Spearman correlation"

    else:
        raise ValueError("method must be 'pearson' or 'spearman'")

    return {
        "method": method_name,
        "n": len(df),
        "correlation": r,
        "p_value": p
    }

## Analyse P-Value for Mortality and LOS

In [ ]:
los_series = encoded_data_df["DurationHospStayDay"]
mortality_series = encoded_data_df["Outcome"].map({"Alive": 0, "Dead": 1})

numerical_features = column_nature_df[column_nature_df["columnNature"] == "numerical"]["columnName"].to_list()
results = []
for feature in numerical_features:
    if feature in encoded_data_df.columns:
        t_stat, _mortality_p_val = pvalue_continuous_vs_binary(encoded_data_df[feature], mortality_series)
        los_p_val = numeric_correlation(encoded_data_df[feature], los_series, method="spearman")["p_value"]
        results.append({
            "feature": feature,
            "mortality_p_value": _mortality_p_val,
            "mortality_significant": _mortality_p_val < 0.05,
            "los_p_value": los_p_val,
            "los_significant": los_p_val < 0.05
        })

results_df = pd.DataFrame(results)
display(results_df)

## All Data

### Compute and Visualise Correlation Matrix

In [ ]:
correlation_df = encoded_data_df.corr(method="spearman", numeric_only=True)
display(correlation_df)

fig = px.imshow(round(correlation_df, 5), text_auto=True, aspect="auto")
fig.show()

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(correlation_df, linewidths=0.5, ax=ax)

In [ ]:
los_series = correlation_df["DurationHospStayDay"]
print(los_series.quantile(0.80))
print(los_series.quantile(0.20))
pd.DataFrame(los_series[(los_series > 0.3) | (los_series < -0.3)].sort_values(ascending=False))

In [ ]:
outcome_series = correlation_df["Outcome"]
print(outcome_series.quantile(0.80))
print(outcome_series.quantile(0.20))
pd.DataFrame(outcome_series[(outcome_series > 0.3) | (outcome_series < -0.3)].sort_values(ascending=False))

## Alive Data

### Compute and Visualise Correlation Matrix

In [ ]:
alive_correlation_df = encoded_alive_data_df.corr(method="spearman")
display(alive_correlation_df)

fig = px.imshow(round(alive_correlation_df, 5), text_auto=True, aspect="auto")
fig.show()

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(alive_correlation_df, linewidths=0.5, ax=ax)

### Alive only LOS correlation

In [ ]:
alive_los_series = alive_correlation_df["DurationHospStayDay"]
print(alive_los_series.quantile(0.80))
print(alive_los_series.quantile(0.20))

pd.DataFrame(alive_los_series[(alive_los_series > 0.3) | (alive_los_series < -0.3)].sort_values(ascending=False))